In [50]:
import json   
import pandas as pd
import numpy as np
from collections import defaultdict
from indicators import get_indicators_for_token
import os
from openpyxl import load_workbook, Workbook
import glob 
from tqdm import tqdm
from pathlib import Path

In [51]:
base = Path("/app/")

In [52]:
token = "2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df"

In [53]:
candles_file = base / "ML_Training_datasets" / "CandleData" / "Candles" / f"{token}_candles.json"
stats_file = base / "ML_Training_datasets" / "CandleData" / "Stats" / f"{token}_stats.json"

In [54]:
with candles_file.open("r", encoding="utf-8") as f:
    candle_data = json.load(f)

with stats_file.open("r", encoding="utf-8") as f:
    stats_data = json.load(f)

In [55]:
token_name = candle_data.get("name", "Unknown")
candles = candle_data.get("timeframes", {}).get("5S", [])

if not candles:
    print("No candles found in JSON.")
else:
    # Convert to DataFrame
    df = pd.DataFrame(candles)

    # Convert timestamp to datetime column (do NOT set as index)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    
    # Sort by timestamp just in case
    df = df.sort_values("timestamp").reset_index(drop=True)

    print(f"Token: {token_name}")


Token: Jellybean


In [56]:
df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume'], dtype='str')

In [57]:
df.head()

,timestamp,open,high,low,close,volume
0,2026-02-27 22:05:45,3.923750e+06,3.926425e+06,3.924924e+06,3.926425e+06,40.206990
1,2026-02-27 22:05:50,3.926425e+06,3.924159e+06,3.919786e+06,3.919786e+06,100.517886
2,2026-02-27 22:06:00,3.919786e+06,3.925031e+06,3.922122e+06,3.925031e+06,78.948570
3,2026-02-27 22:06:05,3.925031e+06,3.981254e+06,3.953001e+06,3.981254e+06,773.203728
4,2026-02-27 22:06:10,3.981254e+06,3.981281e+06,3.977319e+06,3.977319e+06,106.131475


In [58]:
df.shape

(17280, 6)

In [59]:
stats_df = pd.DataFrame(stats_data["stats"])

stats_df["timestamp"] = pd.to_datetime(stats_df["createdAt"])
stats_df = stats_df.sort_values("timestamp")

stats_df = stats_df.drop(columns=["createdAt"])

In [60]:
SUM_COLS = [
    "buyCount",
    "sellCount",
    "buyVolumeSol",
    "sellVolumeSol"
]

MEAN_COLS = ["priceSol"]

In [61]:
WINDOW = 5

stats_5m = stats_df.copy()

stats_5m[SUM_COLS] = (
    stats_df[SUM_COLS]
    .rolling(WINDOW, min_periods=1)
    .sum()
)

stats_5m[MEAN_COLS] = (
    stats_df[MEAN_COLS]
    .rolling(WINDOW, min_periods=1)
    .mean()
)

In [62]:
stats_5m = stats_5m.rename(
    columns={c: f"last_5m_{c}" for c in SUM_COLS + MEAN_COLS}
)

In [63]:
stats_5m.columns

Index(['last_5m_buyCount', 'last_5m_sellCount', 'last_5m_buyVolumeSol',
       'last_5m_sellVolumeSol', 'last_5m_priceSol', 'timestamp'],
      dtype='str')

In [64]:
df["timestamp"] = (
    pd.to_datetime(df["timestamp"], utc=True)
)

stats_5m["timestamp"] = (
    pd.to_datetime(stats_5m["timestamp"], utc=True)
)

In [65]:
df = df.sort_values("timestamp")
stats_5m = stats_5m.sort_values("timestamp")

In [66]:
df["timestamp"] = df["timestamp"].astype("datetime64[ns, UTC]")
stats_5m["timestamp"] = stats_5m["timestamp"].astype("datetime64[ns, UTC]")

In [67]:
df_final = pd.merge_asof(
    df,
    stats_5m,
    on="timestamp",
    direction="backward"
)

In [68]:
df_final.shape

(17280, 11)

In [69]:
df_final.head()

,timestamp,open,high,low,close,volume,last_5m_buyCount,last_5m_sellCount,last_5m_buyVolumeSol,last_5m_sellVolumeSol,last_5m_priceSol
0,2026-02-27 22:05:45+00:00,3.923750e+06,3.926425e+06,3.924924e+06,3.926425e+06,40.206990,NaN,NaN,NaN,NaN,NaN
1,2026-02-27 22:05:50+00:00,3.926425e+06,3.924159e+06,3.919786e+06,3.919786e+06,100.517886,NaN,NaN,NaN,NaN,NaN
2,2026-02-27 22:06:00+00:00,3.919786e+06,3.925031e+06,3.922122e+06,3.925031e+06,78.948570,NaN,NaN,NaN,NaN,NaN
3,2026-02-27 22:06:05+00:00,3.925031e+06,3.981254e+06,3.953001e+06,3.981254e+06,773.203728,NaN,NaN,NaN,NaN,NaN
4,2026-02-27 22:06:10+00:00,3.981254e+06,3.981281e+06,3.977319e+06,3.977319e+06,106.131475,NaN,NaN,NaN,NaN,NaN


In [70]:
def flatten_and_align_indicators(ind_dict, N):
    """
    Normalize indicator outputs so downstream code can safely flatten them.

    - Scalar indicators stay scalar (do NOT expand to lists).
    - List indicators are left-padded/truncated to length N.
    - List[dict] indicators are split into separate list columns.
    """
    aligned = {}

    def pad_list(lst, length, fill=np.nan):
        if lst is None:
            lst = []
        lst = list(lst)
        if len(lst) < length:
            return [fill] * (length - len(lst)) + lst
        return lst[-length:]

    for key, val in ind_dict.items():
        if isinstance(val, list):
            if val and isinstance(val[0], dict):
                subkeys = set()
                for item in val:
                    if isinstance(item, dict):
                        subkeys.update(item.keys())
                for subkey in sorted(subkeys):
                    series = [
                        (item.get(subkey, np.nan) if isinstance(item, dict) else np.nan)
                        for item in val
                    ]
                    aligned[f"{key}_{subkey}"] = pad_list(series, N)
            else:
                aligned[key] = pad_list(val, N)
        else:
            aligned[key] = val

    return aligned

In [71]:
def list_to_rows(df, sequence_mode="latest", max_points=20):
    """
    Flatten indicator list columns without blowing up row-count.

    Parameters
    ----------
    df : pd.DataFrame
    sequence_mode : str
        - "latest": keep only the most recent value from each list column.
        - "window": keep the trailing `max_points` values as separate columns.
        - "explode": legacy behaviour (row explosion), kept only when explicitly needed.
    max_points : int
        Number of trailing points to keep when sequence_mode="window".

    Notes
    -----
    Default mode is "latest" so class distribution is preserved
    (one training row per label event) and memory stays bounded.
    """
    if df.empty:
        return df.copy()

    out = df.copy()
    list_cols = [col for col in out.columns if out[col].map(lambda x: isinstance(x, list)).any()]

    if not list_cols:
        return out.reset_index(drop=True)

    def normalize_list(v):
        if isinstance(v, list):
            return v
        if pd.isna(v):
            return []
        return [v]

    if sequence_mode == "latest":
        for col in list_cols:
            out[col] = out[col].map(lambda v: (normalize_list(v)[-1] if len(normalize_list(v)) else np.nan))
        return out.reset_index(drop=True)

    if sequence_mode == "window":
        max_points = max(1, int(max_points))
        for col in list_cols:
            seq = out[col].map(normalize_list)
            for lag in range(max_points):
                # lag=0 => latest value, lag=1 => previous, ...
                out[f"{col}_lag_{lag}"] = seq.map(
                    lambda s, lag=lag: s[-(lag + 1)] if len(s) > lag else np.nan
                )
            out.drop(columns=[col], inplace=True)
        return out.reset_index(drop=True)

    if sequence_mode == "explode":
        scalar_cols = [col for col in out.columns if col not in list_cols]

        lengths_df = pd.DataFrame(
            {
                col: out[col].map(lambda x: len(x) if isinstance(x, list) else 1)
                for col in list_cols
            },
            index=out.index,
        )
        max_len = lengths_df.max(axis=1).fillna(1).astype(int)

        def pad_list(val, length):
            seq = normalize_list(val)
            if len(seq) < length:
                return [np.nan] * (length - len(seq)) + seq
            return seq[-length:]

        list_part = pd.DataFrame(index=out.index)
        for col in list_cols:
            list_part[col] = [pad_list(v, l) for v, l in zip(out[col], max_len)]

        exploded = list_part.explode(list_cols, ignore_index=False)
        repeated_index = np.repeat(out.index.to_numpy(), max_len.to_numpy())
        scalar_part = out.loc[repeated_index, scalar_cols].reset_index(drop=True)

        exploded = exploded.reset_index(drop=True)
        return pd.concat([scalar_part, exploded], axis=1)

    raise ValueError("sequence_mode must be one of: 'latest', 'window', 'explode'")

In [72]:
def append_to_dataset(flattened_indicators, block_df, dataset, distance, extreme_type, pct_change, 
                      current_close, timeframe, high_wick, low_wick, window_size):
    """
    Enrich flattened indicators with additional features and append to dataset.
    Computes time_to_extreme, relative OHLC, volume stats, momentum, and time features.
    """

    # ---------- Time-to-extreme ----------
    flattened_indicators['time_to_extreme'] = distance * timeframe
    flattened_indicators['time_to_extreme_rel'] = distance / window_size  # relative to window length
    flattened_indicators['extreme_type'] = extreme_type
    flattened_indicators['change_percentage'] = pct_change

    # ---------- Wicks ----------
    flattened_indicators['high_wick'] = high_wick
    flattened_indicators['low_wick'] = low_wick

    # ---------- Volume features ----------
    vol = block_df['volume']
    flattened_indicators['volume_sum'] = vol.sum()
    flattened_indicators['volume_avg'] = vol.mean()
    flattened_indicators['volume_rel'] = vol.mean() / vol.iloc[0]
    flattened_indicators['volume_max_rel'] = vol.max() / vol.mean()

    # ---------- High/Low & Range ----------
    flattened_indicators['rel_high'] = block_df['high'].max() / current_close
    flattened_indicators['rel_low'] = block_df['low'].min() / current_close
    flattened_indicators['range_pct'] = (block_df['high'].max() - block_df['low'].min()) / current_close * 100
    flattened_indicators['subblock_len'] = distance

    # ---------- OHLC relative to first candle ----------
    first_candle = block_df.iloc[0]
    flattened_indicators['open_rel'] = first_candle['open'] / current_close
    flattened_indicators['high_rel'] = first_candle['high'] / current_close
    flattened_indicators['low_rel'] = first_candle['low'] / current_close
    flattened_indicators['close_rel'] = first_candle['close'] / current_close

    # ---------- Short-term momentum ----------
    if len(block_df) > 1:
        flattened_indicators['pct_change_1'] = (first_candle['close'] - first_candle['open']) / current_close * 100
        flattened_indicators['pct_change_2'] = (first_candle['close'] - block_df.iloc[1]['close']) / current_close * 100
    else:
        flattened_indicators['pct_change_1'] = 0.0
        flattened_indicators['pct_change_2'] = 0.0

    # ---------- Time features ----------
    ts = pd.to_datetime(first_candle['timestamp'])
    flattened_indicators['hour'] = ts.hour
    flattened_indicators['minute'] = ts.minute
    flattened_indicators['weekday'] = ts.weekday()
    flattened_indicators['timestamp'] = ts

    dataset.append(flattened_indicators)


In [73]:
def balance_memecoin_dataset_full(df, random_state=42):
    """
    Downsamples all classes in 'extreme_type' to match the smallest class size.
    Ensures equal representation of up, down, and stable.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing 'extreme_type' column
    random_state : int
        Seed for reproducibility

    Returns
    -------
    pd.DataFrame
        Fully balanced dataset
    """
    counts = df['extreme_type'].value_counts()
    print("Before balancing:\n", counts)

    min_count = counts.min()
    dfs = []

    for label in counts.index:
        sampled = df[df['extreme_type'] == label].sample(n=min_count, random_state=random_state)
        dfs.append(sampled)

    df_balanced = pd.concat(dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nAfter balancing:\n", df_balanced['extreme_type'].value_counts())
    return df_balanced


In [74]:
def add_common_features(base_flattened, subblock, current_close, ts_i): 
    base_flattened["volume_sum"] = subblock["volume"].sum() 
    base_flattened["volume_avg"] = subblock["volume"].mean() 
    base_flattened["range_pct"] = (subblock["high"].max() - subblock["low"].min()) / current_close * 100 
    first_candle = subblock.iloc[0] 
    base_flattened["open_rel"] = first_candle["open"] / current_close 
    base_flattened["high_rel"] = first_candle["high"] / current_close 
    base_flattened["low_rel"] = first_candle["low"] / current_close 
    base_flattened["close_rel"] = first_candle["close"] / current_close 
    base_flattened["hour"] = ts_i.hour 
    base_flattened["minute"] = ts_i.minute 
    base_flattened["weekday"] = ts_i.weekday() 
    base_flattened["timestamp"] = ts_i

In [75]:
def create_forward_label_dataset_clean(df, window_size, step_size,
                                       up_threshold, down_threshold,
                                       stable_threshold, timeframe,
                                       history_bars=300,
                                       target_samples=20,
                                       prediction_horizons=(10, 20, 30, 40, 50),
                                       get_indicators_for_token=get_indicators_for_token):
    """
    Build a time-series dataset by iterating once over each absolute candle index.

    - Start at the first candle that has at least `history_bars` prior candles.
    - Compute indicators using up to the most recent `history_bars` ending at that candle.
    - Create fixed-horizon targets for what happened after 10/20/30/40/50 candles.

    Notes
    -----
    `window_size`, `step_size`, `target_samples`, and the threshold arguments are kept
    in the signature for backward compatibility, but this version no longer samples
    inside overlapping windows or builds up/down/stable labels.
    """
    dataset = []

    df_closes = df["close"].values
    df_len = len(df)
    prediction_horizons = tuple(sorted(set(int(h) for h in prediction_horizons if int(h) > 0)))

    if not prediction_horizons:
        raise ValueError("prediction_horizons must contain at least one positive integer")

    max_horizon = max(prediction_horizons)
    start_idx = history_bars
    end_idx = df_len - max_horizon

    if end_idx <= start_idx:
        return pd.DataFrame(dataset)

    total_rows = end_idx - start_idx

    with tqdm(total=total_rows, desc="Rows") as pbar:

        for abs_i in range(start_idx, end_idx):
            current_close = df_closes[abs_i]
            ts_i = pd.to_datetime(df.iloc[abs_i]["timestamp"])

            # Build history subblock (no leakage) using at most the latest `history_bars` bars.
            start_sub = max(0, abs_i - (history_bars - 1))
            subblock = df.iloc[start_sub:abs_i + 1].copy()

            # Compute indicators once
            indicators = get_indicators_for_token(subblock.to_dict("records"))
            base_flattened = flatten_and_align_indicators(indicators, len(subblock))

            # Add common features once to base
            add_common_features(base_flattened, subblock, current_close, ts_i)

            row = base_flattened.copy()
            row["current_close"] = current_close

            for horizon in prediction_horizons:
                future_close = df_closes[abs_i + horizon]
                row[f"future_close_{horizon}"] = future_close
                row[f"future_return_pct_{horizon}"] = ((future_close - current_close) / current_close) * 100.0

            dataset.append(row)
            pbar.update(1)

    df_dataset = pd.DataFrame(dataset)
    df_dataset.fillna(np.nan, inplace=True)
    return df_dataset


In [76]:
dataset = create_forward_label_dataset_clean(
    df=df_final,
    window_size=400,
    step_size=50,
    up_threshold=20.0,
    down_threshold=15.0,
    stable_threshold=10.0,
    timeframe=5,
    history_bars=3000,
    target_samples=100,
    get_indicators_for_token=get_indicators_for_token
)


Rows:   0% 9/14230 [00:22<9:44:34,  2.47s/it] 


KeyboardInterrupt: 

In [28]:
dataset.shape

(14230, 49)

In [29]:
target_cols = [c for c in dataset.columns if c.startswith('future_return_pct_')]
dataset[target_cols].describe()

,future_return_pct_10,future_return_pct_20,future_return_pct_30,future_return_pct_40,future_return_pct_50
count,14230.000000,14230.000000,14230.000000,14230.000000,14230.000000
mean,-0.007956,-0.018802,-0.027349,-0.035023,-0.036961
std,2.408521,3.273223,3.979349,4.611821,5.234262
min,-11.924959,-15.155661,-16.912609,-18.588148,-19.143811
25%,-1.423858,-2.015888,-2.436715,-2.843212,-3.249473
50%,-0.049722,-0.104195,-0.220868,-0.258419,-0.321287
75%,1.321997,1.816787,2.170462,2.466215,2.781424
max,19.872069,30.529658,35.550885,38.126075,40.052123


In [30]:
exploded_dataset = list_to_rows(dataset, sequence_mode="latest")

In [31]:
exploded_dataset.shape

(14230, 49)

In [32]:
target_cols = [c for c in exploded_dataset.columns if c.startswith('future_return_pct_')]
exploded_dataset[target_cols].describe()

,future_return_pct_10,future_return_pct_20,future_return_pct_30,future_return_pct_40,future_return_pct_50
count,14230.000000,14230.000000,14230.000000,14230.000000,14230.000000
mean,-0.007956,-0.018802,-0.027349,-0.035023,-0.036961
std,2.408521,3.273223,3.979349,4.611821,5.234262
min,-11.924959,-15.155661,-16.912609,-18.588148,-19.143811
25%,-1.423858,-2.015888,-2.436715,-2.843212,-3.249473
50%,-0.049722,-0.104195,-0.220868,-0.258419,-0.321287
75%,1.321997,1.816787,2.170462,2.466215,2.781424
max,19.872069,30.529658,35.550885,38.126075,40.052123


In [33]:
exploded_dataset.columns

Index(['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_trend',
       'supertrend_value', 'cci', 'roc', 'momentum3', 'volume_sum',
       'volume_avg', 'range_pct', 'open_rel', 'high_rel', 'low_rel',
       'close_rel', 'hour', 'minute', 'weekday', 'timestamp', 'current_close',
       'future_close_10', 'future_return_pct_10', 'future_close_20',
       'future_return_pct_20', 'future_close_30', 'future_return_pct_30',
       'future_close_40', 'future_return_pct_40', 'future_close_50',
       'future_return_pct_50'],
      dtype='str')

In [34]:
exploded_dataset.head()

,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,future_close_10,future_return_pct_10,future_close_20,future_return_pct_20,future_close_30,future_return_pct_30,future_close_40,future_return_pct_40,future_close_50,future_return_pct_50
0,59.48,3375444.49,3371712.35,3366030.73,3370925.96,3425161.01,1,3852.04,1599.52,2252.52,...,3.368258e+06,-0.547713,3.385576e+06,-0.036361,3.350553e+06,-1.070462,3.339447e+06,-1.398380,3.276727e+06,-3.250271
1,44.97,3372388.98,3370467.29,3365740.86,3370682.66,3424499.10,1,2317.28,1743.07,574.21,...,3.369714e+06,0.329736,3.400547e+06,1.247753,3.356570e+06,-0.061598,3.310790e+06,-1.424657,3.286003e+06,-2.162672
2,35.59,3364749.09,3366648.46,3364353.76,3369884.38,3423562.49,1,-1166.69,1161.12,-2327.81,...,3.371102e+06,1.223056,3.396729e+06,1.992563,3.382616e+06,1.568776,3.311658e+06,-0.561857,3.253908e+06,-2.295891
3,35.53,3358459.99,3363173.28,3363012.79,3369097.74,3422633.10,-1,-3899.80,148.93,-4048.74,...,3.365557e+06,1.062952,3.390333e+06,1.806935,3.389862e+06,1.792802,3.311995e+06,-0.545453,3.250629e+06,-2.388172
4,35.03,3353011.01,3359870.16,3361658.97,3368293.64,3421696.36,-1,-6129.78,-1106.81,-5022.97,...,3.392426e+06,1.920864,3.382887e+06,1.634259,3.388520e+06,1.803491,3.305991e+06,-0.675975,3.217596e+06,-3.331686


In [35]:
exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)

In [36]:
exploded_dataset_clean.shape

(14230, 49)

In [37]:
target_cols = [c for c in exploded_dataset_clean.columns if c.startswith('future_return_pct_')]
exploded_dataset_clean[target_cols].isna().sum()

future_return_pct_10    0
future_return_pct_20    0
future_return_pct_30    0
future_return_pct_40    0
future_return_pct_50    0
dtype: int64

In [38]:
exploded_dataset_clean.head()

,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,future_close_10,future_return_pct_10,future_close_20,future_return_pct_20,future_close_30,future_return_pct_30,future_close_40,future_return_pct_40,future_close_50,future_return_pct_50
0,59.48,3375444.49,3371712.35,3366030.73,3370925.96,3425161.01,1,3852.04,1599.52,2252.52,...,3.368258e+06,-0.547713,3.385576e+06,-0.036361,3.350553e+06,-1.070462,3.339447e+06,-1.398380,3.276727e+06,-3.250271
1,44.97,3372388.98,3370467.29,3365740.86,3370682.66,3424499.10,1,2317.28,1743.07,574.21,...,3.369714e+06,0.329736,3.400547e+06,1.247753,3.356570e+06,-0.061598,3.310790e+06,-1.424657,3.286003e+06,-2.162672
2,35.59,3364749.09,3366648.46,3364353.76,3369884.38,3423562.49,1,-1166.69,1161.12,-2327.81,...,3.371102e+06,1.223056,3.396729e+06,1.992563,3.382616e+06,1.568776,3.311658e+06,-0.561857,3.253908e+06,-2.295891
3,35.53,3358459.99,3363173.28,3363012.79,3369097.74,3422633.10,-1,-3899.80,148.93,-4048.74,...,3.365557e+06,1.062952,3.390333e+06,1.806935,3.389862e+06,1.792802,3.311995e+06,-0.545453,3.250629e+06,-2.388172
4,35.03,3353011.01,3359870.16,3361658.97,3368293.64,3421696.36,-1,-6129.78,-1106.81,-5022.97,...,3.392426e+06,1.920864,3.382887e+06,1.634259,3.388520e+06,1.803491,3.305991e+06,-0.675975,3.217596e+06,-3.331686


In [39]:
df_new = exploded_dataset_clean.copy()

In [40]:
df_new.shape

(14230, 49)

In [41]:
target_cols = [c for c in df_new.columns if c.startswith('future_return_pct_')]
df_new[target_cols].describe()

,future_return_pct_10,future_return_pct_20,future_return_pct_30,future_return_pct_40,future_return_pct_50
count,14230.000000,14230.000000,14230.000000,14230.000000,14230.000000
mean,-0.007956,-0.018802,-0.027349,-0.035023,-0.036961
std,2.408521,3.273223,3.979349,4.611821,5.234262
min,-11.924959,-15.155661,-16.912609,-18.588148,-19.143811
25%,-1.423858,-2.015888,-2.436715,-2.843212,-3.249473
50%,-0.049722,-0.104195,-0.220868,-0.258419,-0.321287
75%,1.321997,1.816787,2.170462,2.466215,2.781424
max,19.872069,30.529658,35.550885,38.126075,40.052123


In [42]:
df_new.head()

,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,future_close_10,future_return_pct_10,future_close_20,future_return_pct_20,future_close_30,future_return_pct_30,future_close_40,future_return_pct_40,future_close_50,future_return_pct_50
0,59.48,3375444.49,3371712.35,3366030.73,3370925.96,3425161.01,1,3852.04,1599.52,2252.52,...,3.368258e+06,-0.547713,3.385576e+06,-0.036361,3.350553e+06,-1.070462,3.339447e+06,-1.398380,3.276727e+06,-3.250271
1,44.97,3372388.98,3370467.29,3365740.86,3370682.66,3424499.10,1,2317.28,1743.07,574.21,...,3.369714e+06,0.329736,3.400547e+06,1.247753,3.356570e+06,-0.061598,3.310790e+06,-1.424657,3.286003e+06,-2.162672
2,35.59,3364749.09,3366648.46,3364353.76,3369884.38,3423562.49,1,-1166.69,1161.12,-2327.81,...,3.371102e+06,1.223056,3.396729e+06,1.992563,3.382616e+06,1.568776,3.311658e+06,-0.561857,3.253908e+06,-2.295891
3,35.53,3358459.99,3363173.28,3363012.79,3369097.74,3422633.10,-1,-3899.80,148.93,-4048.74,...,3.365557e+06,1.062952,3.390333e+06,1.806935,3.389862e+06,1.792802,3.311995e+06,-0.545453,3.250629e+06,-2.388172
4,35.03,3353011.01,3359870.16,3361658.97,3368293.64,3421696.36,-1,-6129.78,-1106.81,-5022.97,...,3.392426e+06,1.920864,3.382887e+06,1.634259,3.388520e+06,1.803491,3.305991e+06,-0.675975,3.217596e+06,-3.331686


In [49]:
out_path = base / "ML_Training_datasets" / "Datasets" / "5S" / "memecoin_training_dataset_1.csv"
df_new.to_csv(out_path, index=False)

/app/ML_Training_datasets/Datasets/5S/memecoin_training_dataset_1.csv
